In [1]:
import warnings, osmnx
warnings.filterwarnings("ignore,")


ModuleNotFoundError: No module named 'pandas._libs.pandas_parser'

**Importing necessary packages**

In [2]:
import pandas
import osmnx
import geopandas
import rioxarray
import xarray
import datashader
import contextily as cx
from shapely import geometry
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'pandas._libs.pandas_parser'

**Check your current working directory:**

In [ ]:
import os

# Get the current working directory
current_directory = os.getcwd()

print("Current working directory:", current_directory)

Current working directory: /Users/Adnan/Siddique/Work-Related/Teaching/ITU/Spatial Data Science/Spring 2025/code-sds-sp2025/Notebooks


# Part I

## Fundamental geographic data structures

There are a few main data structures that are used in geographic data science: 
- geographic tables (which are generally matched to an object data model), 
- rasters or surfaces (which are generally matched to a field data model), and 
- spatial networks (which are generally matched to a graph data model). 

### Geographic tables

Geographic objects are usually matched to what we called the *geographic table*. Geographic tables can be thought of as a tab in a spreadsheet where one of the columns records geometric information. This data structure represents a single geographic object as a row of a table; each column in the table records information about the object, its attributes or features, as we will see below. Typically, there is a special column in this table that records the *geometry* of the object.

To understand the structure of geographic tables, it will help to read in the `countries_clean.gpkg` dataset included in this course that describes countries in the world. To read in this data, we can use the `read_file()` method in `geopandas`:

In [ ]:
gt_polygons = geopandas.read_file(
    "...")

And we can examine the top of the table with the `.head()` method:

In [ ]:
gt_polygons.head()

<div class="alert alert-info">
<span style="font-size:1.5em;">&#9432;</span> 
<strong> What are these `MULTIPOLYGON` and `POLYGON` objects?</strong>
Just continue for now, and we'll discuss them in detail in Part II.
</div>

Geographic tables store geographic information as an additional column. But, how is this information encoded? To see, we can check the type of the object in the first row:

In [ ]:
type(gt_polygons.geometry[0])

In `geopandas` (as well as other packages representing geographic data), the `geometry` column has special traits which a "normal" column, such as `ADMIN`, does not. For example, when we plot the dataframe, the `geometry` column is used as the main shape to use in the plot.

In [ ]:
gt_polygons.plot()

Changing the geometric representation of a sample must be done carefully: since the `geometry` column is special, there are special functions to adjust the geometry. For example, if we wanted to represent each country using its *centroid*, a point in the middle of the shape, then we must take care to make sure that a new geometry column was set properly using the `set_geometry()` method. This can be useful when you want to work with two different geometric representations of the same underlying sample. 

Let us make a map of both the boundary and the centroid of a country. First, to compute the centroid, we can use the `gt_polygons.geometry.centroid` property. This gives us the point that minimizes the average distance from all other points on the boundary of the shape. Storing that back to a column, called `centroid`:

In [ ]:
gt_polygons["centroid"] = gt_polygons...

We now have an additional feature:

In [ ]:
gt_polygons.head()

Despite the fact that `centroid` is a geometry (you can tell because each cell starts with `POINT`), it is not currently set as the active geometry for our table. We can switch to the `centroid` column using the `set_geometry()` method. Finally, we can plot the centroid and the boundary of each country after switching the geometry column with `set_geometry()`:

In [ ]:
# Plot centroids
ax = gt_polygons.set_geometry("centroid").plot("ADMIN", markersize=5)
# Plot polygons without color filling
gt_polygons.plot(
    "ADMIN", ax=ax, facecolor="none", edgecolor= ..., linewidth=0.2
);

Note again how we can create a map by calling `.plot()` on a `GeoDataFrame`. We can thematically color each feature based on a column by passing the name of that column to the plot method (as we do on with `ADMIN` in this case), and that the current geometry is used.

Thus, as should now be clear, nearly any kind of geographic object can be represented in one (or more) geometry column(s). Thinking about the number of different kinds of shapes or geometries one could use quickly boggles the mind. Fortunately the Open Geospatial Consortium (OGC) has defined a set of "abstract" types that can be used to define any kind of geometry. This specification, codified in ISO 19125-1---the "simple features" specification---defines the formal relationships between these types: a `Point` is a zero-dimensional location with an x and y coordinate, a `LineString` is a path composed of a set of more than one `Point`, and a `Polygon` is a surface that has at least one `LineString` that starts and stops with the same coordinate. All of these types *also* have `Multi` variants that indicate a collection of multiple geometries of the same type. So, for instance, Bolivia is represented as a single polygon:

In [ ]:
gt_polygons.query('ADMIN == "Bolivia"')

In [ ]:
# plot Bolivia
gt_polygons.query('ADMIN == "Bolivia"')...

while Indonesia is a `MultiPolygon` containing many `Polygons` for each individual island in the country:

In [ ]:
gt_polygons.query('ADMIN == "Indonesia"')

In [ ]:
gt_polygons.query('ADMIN == "Indonesia"').plot();

:question: Query & then plot Pakistan.

In [ ]:
# Your code here

In many cases, geographic tables will have geometries of a single type; records will *all* be `Point` or `LineString`, for instance. However, there is no formal requirement that a *geographic table* has geometries that all have the same type. 

Throughout this book, we will use geographic tables extensively, storing polygons, but also points and lines. We will explore lines a bit more in the second part of this chapter but, for now, let us stop on points for a second. As mentioned above, these are the simplest type of feature in that they do not have any dimension, only a pair of coordinates attached to them. This means that points can sometimes be stored in a non-geographic table, simply using one column for each coordinate. We find an example of this on the Tokyo dataset we will use more later. The data is stored as a comma-separated value table, or `.csv`:

In [ ]:
gt_points = pandas.read_csv("../Course-Datasets/tokyo_clean.csv")

Since we have read it with `pandas`, the table is loaded as a `DataFrame`, with no explicit spatial dimension:

In [ ]:
type(gt_points)

If we inspect the table, we find there is not a `geometry` column:

In [ ]:
gt_points.head()

Many point datasets are provided in this format. To make the most of them, it is convenient to convert them into `GeoDataFrame` tables. There are two steps involved in this process. First, we turn the raw coordinates into geometries:

In [ ]:
pt_geoms = geopandas.points_from_xy(
    x=gt_points["longitude"],
    y=gt_points["latitude"],
    # x,y are Earth longitude & latitude
    crs="EPSG:4326",
)

Second, we create a `GeoDataFrame` object using these geometries:

In [ ]:
gt_points = geopandas.GeoDataFrame(gt_points, geometry=pt_geoms)

And now `gt_points` looks and feels exactly like the one of countries we have seen before, with the difference the `geometry` column stores `POINT` geometries:

In [ ]:
gt_points.head()

--- 

In [ ]:
# Check the number of rows & columsn for the dataset using .shape method
nrows_gt_points = ...
ncolumns_gt_points = ...
print(f"The number of rows in the dataframe are: {nrows_gt_points}")
print(f"The number of rows in the dataframe are: {ncolumns_gt_points}")

In [ ]:
# Choose a subset of the `gt_points` geoDataFrame; keep all columns though
gt_points_subset = ...

:question: Plot the dataset `gt_points_subset` with a basemap
> Use contextily.

In [ ]:
# Your code here!

## Find a UTM zone appropriate for the geoDataFrame `gt_points_subset`
> Use the method .estimate_utm_crs()

In [ ]:
# Your code here.

## Project the CRS of `gt_points_subset` to the UTM zone found above

In [ ]:
# Your code here.

## Plot the projected data frame

In [ ]:
# Your code here

:question: Do you observe any change?

Your answer here.

:question: Where can this approach fail?

Your answer here.